# Aula 3 · Classificação

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AlexChequer/notebooks-insperai/blob/main/trainees/aula-03-classificacao.ipynb)

**Trilha de Trainees — InsperAI** · o par prático da [Aula 3](https://trilhas-insperai.vercel.app/trainees/aulas/aula-03/)

---

**Este notebook é diferente dos anteriores.** Nas Aulas 1 e 2 o código já vinha
pronto e você rodava para ver o efeito. Aqui **você escreve**. São 8 funções para
implementar, cada uma com uma célula de teste logo abaixo que diz na hora se
está certo.

Regressão logística é o primeiro modelo que dá para construir inteiro do zero em
meia hora — sigmoid, custo, gradiente, treino — e é o esqueleto de tudo que vem
depois. A Aula 5 vai empilhar isto em camadas e chamar de rede neural. Vale a
pena ter escrito com as próprias mãos uma vez.

**Como funciona.** Onde tem `### SEU CÓDIGO AQUI ###`, a função levanta um erro
até você implementar — `Run all` não resolve nada aqui. A célula de teste abaixo
imprime o que passou e o que falhou, com o número que veio e o que era esperado.
Cada exercício tem a resposta num `<details>` recolhido: use depois de tentar, e
leia o *porquê* mesmo quando acertar.

O dado é o **Breast Cancer Wisconsin**, que vem dentro do scikit-learn — 569
tumores, 30 medidas de cada um, e o diagnóstico. É o mesmo exemplo que a página
usa para explicar por que recall importa mais que acurácia num diagnóstico.


## 0. Setup — rode e siga

Duas coisas aqui: o dado, e o `verificar()`, que é o que as células de teste usam
para imprimir o resultado. Você não precisa entender o `verificar()` — mas ele é
curto, se der curiosidade.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer

np.set_printoptions(precision=4, suppress=True)


def verificar(titulo, *casos):
    """Imprime o resultado de uma bateria de testes.

    Cada caso é (descrição, obtido, esperado) para número/array, ou
    (descrição, bool) para uma condição que já foi avaliada.

    Mostra todas as falhas de uma vez, de propósito: parar no primeiro erro
    esconde que os outros três também estão errados pelo mesmo motivo.
    """
    linhas, falhas = [], 0
    for caso in casos:
        if len(caso) == 2:
            desc, ok = caso
            detalhe = ""
        else:
            desc, obtido, esperado = caso
            ok = np.allclose(np.asarray(obtido, dtype=float),
                             np.asarray(esperado, dtype=float),
                             rtol=1e-4, atol=1e-6)
            detalhe = "" if ok else f"   → veio {np.asarray(obtido)}, esperava {np.asarray(esperado)}"
        falhas += not ok
        linhas.append(f"  {'✓' if ok else '✗'} {desc}{detalhe}")
    print("\n".join(linhas))
    if falhas:
        print(f"\n✗ {titulo}: {falhas} de {len(casos)} falharam. Olhe as linhas com ✗.")
    else:
        print(f"\n✓ {titulo}: passou nos {len(casos)} testes.")


# O dado. Cuidado com uma pegadinha do sklearn: lá, target=0 é maligno e
# target=1 é benigno. Invertemos, porque a classe que interessa (a que queremos
# "pegar") tem que ser a classe 1 — é a convenção de todas as métricas.
dados = load_breast_cancer()
y_todos = 1 - dados.target                      # 1 = maligno, 0 = benigno
COLUNAS = ["mean radius", "mean texture"]       # duas features, para dar pra ver no plano
idx = [list(dados.feature_names).index(c) for c in COLUNAS]
X_todos = dados.data[:, idx]

print(f"{X_todos.shape[0]} tumores, {X_todos.shape[1]} features: {COLUNAS}")
print(f"malignos (classe 1): {y_todos.sum()}   benignos (classe 0): {(y_todos == 0).sum()}")
print(f"proporção de malignos: {y_todos.mean():.1%}")


Separando treino e teste, e padronizando — as duas coisas da Aula 2. Repare que
média e desvio saem **só do treino**: usar o teste para calcular a escala é
deixar o teste vazar para dentro do modelo, e foi um quiz inteiro da Aula 2.


In [ ]:
rng = np.random.default_rng(0)
ordem = rng.permutation(len(y_todos))
corte = int(0.75 * len(y_todos))
tr, te = ordem[:corte], ordem[corte:]

media, desvio = X_todos[tr].mean(axis=0), X_todos[tr].std(axis=0)
X_treino, y_treino = (X_todos[tr] - media) / desvio, y_todos[tr]
X_teste,  y_teste  = (X_todos[te] - media) / desvio, y_todos[te]

print(f"treino: {X_treino.shape[0]} exemplos   teste: {X_teste.shape[0]} exemplos")

fig, ax = plt.subplots(figsize=(5.5, 4.2))
for classe, cor, nome in [(0, "tab:blue", "benigno"), (1, "tab:red", "maligno")]:
    m = y_treino == classe
    ax.scatter(X_treino[m, 0], X_treino[m, 1], s=14, c=cor, label=nome, alpha=0.7)
ax.set_xlabel(f"{COLUNAS[0]} (padronizado)"); ax.set_ylabel(f"{COLUNAS[1]} (padronizado)")
ax.set_title("Os dois grupos, nas duas features"); ax.legend()
plt.tight_layout(); plt.show()


## 1. Por que a reta quebra

A página abre com isto e vale ver acontecer no dado real. A tentação é tratar a
classe como número (0 ou 1) e usar a regressão linear da Aula 1.

Abaixo, uma feature só (`mean radius`), a reta ajustada, e a **fronteira**: o
ponto onde a reta cruza 0,5 e a decisão vira de benigno para maligno.


In [ ]:
from sklearn.linear_model import LinearRegression

x1 = X_treino[:, [0]]

def fronteira_da_reta(x, y):
    """Onde a reta ajustada cruza 0,5 — o ponto em que a decisão muda."""
    reta = LinearRegression().fit(x, y)
    return (0.5 - reta.intercept_) / reta.coef_[0], reta

corte_original, reta = fronteira_da_reta(x1, y_treino)

# Agora um único tumor absurdo, muito à direita e obviamente maligno.
# MUDE AQUI: empurre o 12 para 30, 60, 100 e veja a fronteira andar.
POSICAO_DO_OUTLIER = 12.0

x_out = np.vstack([x1, [[POSICAO_DO_OUTLIER]]])
y_out = np.append(y_treino, 1)
corte_novo, reta_nova = fronteira_da_reta(x_out, y_out)

print(f"fronteira sem o outlier: {corte_original:6.3f}")
print(f"fronteira com o outlier: {corte_novo:6.3f}")
print(f"ela andou {abs(corte_novo - corte_original):.3f} desvios por causa de UM ponto")

grade = np.linspace(-2.5, max(3.5, POSICAO_DO_OUTLIER + 1), 200).reshape(-1, 1)
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.scatter(x1, y_treino, s=12, c="k", alpha=0.35, label="tumores")
ax.scatter([POSICAO_DO_OUTLIER], [1], s=80, c="tab:red", zorder=5, label="o outlier")
ax.plot(grade, reta.predict(grade), c="tab:blue", label="reta sem o outlier")
ax.plot(grade, reta_nova.predict(grade), c="tab:orange", label="reta com o outlier")
ax.axhline(0.5, ls=":", c="gray")
ax.axvline(corte_original, ls="--", c="tab:blue", alpha=0.6)
ax.axvline(corte_novo, ls="--", c="tab:orange", alpha=0.6)
ax.set_ylim(-0.6, 1.6); ax.set_xlabel("mean radius (padronizado)"); ax.set_ylabel("classe")
ax.set_title("Um ponto distante gira a reta e move a fronteira"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


Dois problemas, os mesmos que a página descreve: a reta **passa de 1 e desce
abaixo de 0** (não existe "1,7 de maligno"), e **um ponto só move a fronteira**,
porque o erro quadrático paga caro por aquele exemplo distante e prefere girar a
reta inteira a deixá-lo errado.

O conserto é fazer a saída caber entre 0 e 1. É a sigmoid.


## 2. A sigmoid, e de onde ela sai

A página apresenta a fórmula pronta. Aqui vale ver **de onde ela vem**, porque
ela não é um chute que por acaso deu certo.

Queremos uma probabilidade $p \in (0, 1)$, mas a conta que sabemos fazer,
$z = \mathbf{w} \cdot \mathbf{x} + b$, devolve qualquer real. Precisamos de uma
ponte entre os dois mundos. O caminho clássico é ir do lado apertado para o lado
largo e depois inverter:

**Passo 1 — as chances.** Em vez da probabilidade, use a razão de chances:

$$\text{chances} = \frac{p}{1-p}$$

Isso já ajuda: $p \in (0,1)$ vira $\text{chances} \in (0, \infty)$. Metade do
caminho — o teto sumiu, o piso em zero continua.

**Passo 2 — o log.** O logaritmo estica $(0, \infty)$ para $(-\infty, \infty)$:

$$\log\frac{p}{1-p} \in (-\infty, \infty)$$

Agora sim: o lado esquerdo mora no mesmo lugar que o $z$. Essa quantidade tem
nome, **log-odds**, e a aposta do modelo é que *ela* é linear nas features:

$$\log\frac{p}{1-p} = z$$

**Passo 3 — isolar o $p$.** É só álgebra:

$$\frac{p}{1-p} = e^{z} \;\Longrightarrow\; p = e^{z} - p\,e^{z} \;\Longrightarrow\; p\,(1 + e^{z}) = e^{z} \;\Longrightarrow\; p = \frac{e^{z}}{1 + e^{z}}$$

E dividindo em cima e embaixo por $e^{z}$:

$$\boxed{\;p = \sigma(z) = \frac{1}{1 + e^{-z}}\;}$$

A sigmoid não foi escolhida por ter cara bonita: ela é **a função que inverte o
log-odds**. Toda vez que você assume que o log das chances é linear, ela cai no
seu colo.

Três propriedades que os testes vão cobrar, e que valem na cabeça:

| Propriedade | Por quê |
|---|---|
| $\sigma(0) = \tfrac12$ | $e^{0}=1$, então $1/(1+1)$ |
| $\sigma(-z) = 1 - \sigma(z)$ | simetria: a chance de ser 0 é o espelho da de ser 1 |
| $\sigma'(z) = \sigma(z)\,(1-\sigma(z))$ | a derivada sai dela mesma — é o que deixa o gradiente tão limpo |


### Exercício 1 — `sigmoid(z)`

Implemente $\sigma(z) = \dfrac{1}{1 + e^{-z}}$.

Ela precisa funcionar com **um número e com um array** — todo o resto do
notebook vai chamá-la com os 426 exemplos de uma vez. Use `np.exp`, que já opera
elemento a elemento; se você escrever a conta sem `for`, ganha isso de graça.


In [ ]:
def sigmoid(z):
    """A sigmoid, elemento a elemento.

    z : número ou np.ndarray de qualquer formato
    retorna: mesmo formato de z, com valores em (0, 1)
    """
    z = np.asarray(z, dtype=float)
    ### SEU CÓDIGO AQUI ###  (≈ 1 linha)
    raise NotImplementedError("Apague esta linha e devolva a sigmoid de z.")


In [ ]:
verificar(
    "Exercício 1 · sigmoid",
    ("sigmoid(0) = 0,5", sigmoid(0), 0.5),
    ("satura em ~1 para z grande", sigmoid(50), 1.0),
    ("satura em ~0 para z muito negativo", sigmoid(-50), 0.0),
    ("sigmoid(2) = 0,880797", sigmoid(2.0), 0.8807970779778823),
    ("simetria: sigmoid(-z) = 1 - sigmoid(z)", sigmoid(-1.3), 1 - sigmoid(1.3)),
    ("funciona com array", sigmoid(np.array([-2.0, 0.0, 2.0])),
     [0.1192029220221175, 0.5, 0.8807970779778823]),
    ("a saída fica dentro de (0, 1)",
     bool(np.all((sigmoid(np.linspace(-30, 30, 500)) > 0)
                 & (sigmoid(np.linspace(-30, 30, 500)) < 1)))),
)


<details><summary>Resposta do Exercício 1</summary>

```python
    return 1.0 / (1.0 + np.exp(-z))
```

Uma linha, e o `np.exp` cuida do array inteiro sozinho — nenhum `for` precisa
existir. Se você escreveu um laço, funciona, mas vale reescrever sem: o resto do
notebook chama a sigmoid com 426 exemplos de uma vez, e a versão vetorizada é a
que você vai ver em qualquer código de verdade.

**Uma armadilha que os testes não pegam.** Para `z` muito negativo — digamos
−1000 —, `np.exp(1000)` estoura e o NumPy avisa `overflow`. A conta ainda dá o
resultado certo (`1/inf = 0`), mas o aviso aparece. A versão à prova de disso
trata os dois lados separado:

```python
positivo = z >= 0
saida = np.empty_like(z, dtype=float)
saida[positivo] = 1.0 / (1.0 + np.exp(-z[positivo]))
exp_z = np.exp(z[~positivo])
saida[~positivo] = exp_z / (1.0 + exp_z)
```

É o que bibliotecas de verdade fazem. Para este notebook a versão de uma linha
basta — mas agora você sabe por que a de verdade é feia.

**E um detalhe que o teste quase pegou:** ele varre até ±30, não mais. A partir
de ±37 ou assim, `sigmoid(z)` devolve **exatamente 1.0** — não "quase 1". O
float64 tem uns 16 dígitos, e `1 − 8e-18` arredonda para 1 na hora de guardar.
É por isso que o Exercício 3 vai precisar de um `clip`: a saturação aqui é real,
não teórica.

</details>


In [ ]:
z = np.linspace(-8, 8, 300)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(z, sigmoid(z), c="tab:green")
ax[0].axhline(0.5, ls=":", c="gray"); ax[0].axvline(0, ls=":", c="gray")
ax[0].set_title("σ(z)"); ax[0].set_xlabel("z")
ax[1].plot(z, sigmoid(z) * (1 - sigmoid(z)), c="tab:purple")
ax[1].set_title("σ'(z) = σ(z)·(1−σ(z))"); ax[1].set_xlabel("z")
plt.tight_layout(); plt.show()

print("Repare no gráfico da direita: a derivada morre nas duas pontas.")
print("Guarde isso — é o assunto do 'gradiente que some' da Aula 5.")


## 3. O modelo inteiro

Com a sigmoid pronta, o modelo é uma linha de conta: o produto escalar da Aula 2,
passado pela sigmoid.

$$p = \sigma(\mathbf{w} \cdot \mathbf{x} + b)$$

### Exercício 2 — `prever_probabilidade(X, w, b)`

`X` tem um exemplo por linha e uma feature por coluna (426 × 2, aqui). `w` é um
vetor com um peso por feature. Devolva um vetor com **uma probabilidade por
exemplo**.

Dica: é `X @ w + b` por dentro, como no desafio da revisão — e depois a sigmoid.


In [ ]:
def prever_probabilidade(X, w, b):
    """Probabilidade de cada exemplo ser da classe 1.

    X : (n_exemplos, n_features)
    w : (n_features,)
    b : número
    retorna: (n_exemplos,) com valores em (0, 1)
    """
    ### SEU CÓDIGO AQUI ###  (≈ 1 linha)
    raise NotImplementedError("Apague esta linha e devolva as probabilidades.")


In [ ]:
X_p = np.array([[1.0, 2.0], [0.0, 0.0], [-1.0, -1.0]])
w_p, b_p = np.array([0.5, -0.25]), 0.1

verificar(
    "Exercício 2 · prever_probabilidade",
    ("devolve um valor por exemplo", np.shape(prever_probabilidade(X_p, w_p, b_p)), (3,)),
    ("com w e b zerados, tudo é 0,5",
     prever_probabilidade(X_p, np.zeros(2), 0.0), [0.5, 0.5, 0.5]),
    ("o b sozinho desloca todo mundo",
     prever_probabilidade(X_p, np.zeros(2), 2.0), [sigmoid(2.0)] * 3),
    ("os três valores certos",
     prever_probabilidade(X_p, w_p, b_p),
     [0.5249791875, 0.5249791875, 0.4625702017]),
    ("roda com o dado real",
     np.shape(prever_probabilidade(X_treino, np.zeros(2), 0.0)), (len(y_treino),)),
)


<details><summary>Resposta do Exercício 2</summary>

```python
    return sigmoid(X @ w + b)
```

O `@` é o produto escalar de cada linha de `X` com `w`, para as 426 linhas de uma
vez. Se você usou `np.dot(X, w)`, é a mesma coisa.

**O erro mais comum aqui** é escrever `w @ X`. Com `X` de formato (426, 2) e `w`
de (2,), isso não alinha e o NumPy reclama. A ordem importa: as features de `X`
estão nas **colunas**, então `w` multiplica pela direita.

</details>


## 4. O custo: por que não dá para usar o MSE

A página diz que aqui a função de custo é a **cross-entropy**, e que o MSE cria
"mínimos locais e gradiente quase morto nas pontas". Vamos **ver** isso antes de
implementar, porque é a única parte da aula em que a justificativa é visual.


In [ ]:
# Uma fatia da superfície de custo: varia só w[0], com o resto fixo.
def mse(X, y, w, b):
    return ((prever_probabilidade(X, w, b) - y) ** 2).mean()

def cross_entropy_referencia(X, y, w, b):
    p = np.clip(prever_probabilidade(X, w, b), 1e-12, 1 - 1e-12)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p)).mean()

# Um problema pequeno e propositalmente desconfortável, para o MSE mostrar o defeito
Xd = np.array([[-2.0], [-1.0], [1.0], [2.0], [6.0]])
yd = np.array([0.0, 0.0, 1.0, 1.0, 1.0])
faixa = np.linspace(-6, 12, 400)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(faixa, [mse(Xd, yd, np.array([w]), 0.0) for w in faixa], c="tab:red")
ax[0].set_title("MSE + sigmoid"); ax[0].set_xlabel("w")
ax[1].plot(faixa, [cross_entropy_referencia(Xd, yd, np.array([w]), 0.0) for w in faixa], c="tab:green")
ax[1].set_title("cross-entropy"); ax[1].set_xlabel("w")
for a in ax: a.set_ylabel("custo")
plt.tight_layout(); plt.show()

print("Esquerda: platôs quase planos nas duas pontas — o gradiente lá é ~0 e o")
print("treino não sai do lugar, mesmo estando longe do mínimo.")
print("Direita: uma tigela. Sempre há uma descida apontando para o fundo.")


A da direita é uma tigela — **convexa**, no jargão: qualquer ponto de partida
desce para o mesmo fundo. A da esquerda tem regiões planas onde o gradiente é
praticamente zero, e o gradient descent trava nelas achando que chegou.

**De onde vem a cross-entropy.** Ela também não é um chute. Se $p$ é a
probabilidade que o modelo dá para a classe 1, a probabilidade que ele atribui ao
rótulo que **realmente aconteceu** é

$$p \;\text{ se } y=1, \qquad 1-p \;\text{ se } y=0$$

que dá para escrever num só termo, sem `if`:

$$p^{\,y}\,(1-p)^{1-y}$$

(confira: com $y=1$ o segundo fator vira $(1-p)^0 = 1$; com $y=0$ o primeiro
vira $p^0 = 1$.)

Queremos os pesos que tornam o observado o mais provável possível. Multiplicar
centenas dessas probabilidades dá um número minúsculo e instável, então tomamos
o log — que transforma produto em soma —, e trocamos o sinal para virar algo a
**minimizar** em vez de maximizar:

$$J(\mathbf{w}, b) = -\frac{1}{m}\sum_{i=1}^{m}\Big[\, y^{(i)}\log p^{(i)} \;+\; (1-y^{(i)})\log\big(1-p^{(i)}\big) \Big]$$

Onde está a "punição da confiança errada" que a página menciona: se o rótulo é 1
e o modelo disse $p = 0{,}01$, o termo é $-\log(0{,}01) \approx 4{,}6$. Se disse
$p = 0{,}4$, é só $0{,}92$. Errar com convicção custa cinco vezes mais — e quando
$p \to 0$ o custo vai a infinito.

### Exercício 3 — `custo(X, y, w, b)`

Implemente a fórmula acima.

**Um detalhe que quebra na prática:** se a sigmoid saturar e devolver exatamente
0 ou 1, o `np.log` devolve `-inf` e o custo vira `nan`. Use
`np.clip(p, 1e-12, 1 - 1e-12)` antes do log. Não é preciosismo — acontece já no
treino desta aula.


In [ ]:
def custo(X, y, w, b):
    """Cross-entropy média sobre os m exemplos.

    retorna: um número
    """
    ### SEU CÓDIGO AQUI ###  (≈ 3 linhas)
    raise NotImplementedError("Apague esta linha e devolva o custo.")


In [ ]:
X_c = np.array([[1.0, 2.0], [0.0, 0.0], [-1.0, -1.0], [2.0, 1.0]])
y_c = np.array([1.0, 0.0, 0.0, 1.0])

# Com w=0 e b=0 todas as probabilidades são 0,5, então o custo é -log(0,5)
esperado_chute = -np.log(0.5)

# Um modelo quase perfeito custa quase nada; um confiante e errado custa caro
X_um = np.array([[10.0]])
verificar(
    "Exercício 3 · custo",
    ("devolve um número, não um array", np.ndim(custo(X_c, y_c, np.zeros(2), 0.0)), 0),
    ("chutando 0,5 em tudo, o custo é log(2) = 0,6931",
     custo(X_c, y_c, np.zeros(2), 0.0), esperado_chute),
    ("acertar com confiança custa quase nada",
     custo(X_um, np.array([1.0]), np.array([1.0]), 0.0), 4.539889921686465e-05),
    ("errar com confiança custa caro",
     custo(X_um, np.array([0.0]), np.array([1.0]), 0.0), 10.000045398899218),
    ("caso com dois pesos",
     custo(X_c, y_c, np.array([0.5, -0.25]), 0.1), 0.5914038590947175),
    ("não vira nan quando a sigmoid satura",
     bool(np.isfinite(custo(np.array([[100.0]]), np.array([0.0]), np.array([5.0]), 0.0)))),
)


<details><summary>Resposta do Exercício 3</summary>

```python
    p = prever_probabilidade(X, w, b)
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
```

O truque do `y * log(p) + (1-y) * log(1-p)` é o `if` escrito como conta: quando
`y` é 1 o segundo termo zera, quando é 0 o primeiro zera. Isso não é elegância
gratuita — é o que permite rodar nos 426 exemplos de uma vez, com `y` como vetor.

**Sobre o `clip`:** o último teste tem `z = 500`, e `sigmoid(500)` devolve
exatamente `1.0` (não "quase 1" — o float acabou). Com o rótulo 0, o termo vira
`log(1 - 1.0) = log(0) = -inf`, e o custo é `inf`. O `clip` troca isso pelo
custo altíssimo mas finito de ~27,6, que é o que o gradient descent consegue
usar. Sem ele o treino morre com `nan` e nada mais faz sentido.

</details>


## 5. O gradiente — e a surpresa

Para treinar, precisamos das derivadas do custo. A conta é chata mas o resultado
é uma das coisas mais bonitas do curso. Em resumo, usando
$\sigma'(z) = \sigma(z)(1-\sigma(z))$ e a regra da cadeia, os termos se cancelam
e sobra:

$$\frac{\partial J}{\partial w_j} = \frac{1}{m}\sum_{i=1}^{m}\big(p^{(i)} - y^{(i)}\big)\,x_j^{(i)}
\qquad
\frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=1}^{m}\big(p^{(i)} - y^{(i)}\big)$$

**Compare com a Aula 1.** Lá o gradiente era $\frac{1}{m}\sum(\hat{y} - y)\,x$ —
**exatamente a mesma fórmula**. Modelo diferente, custo diferente, e a expressão
do gradiente é idêntica; só muda o que está dentro do $p$.

Isso não é coincidência: cross-entropy e sigmoid foram feitas uma para a outra, e
o $\sigma(1-\sigma)$ da derivada da sigmoid cancela com o denominador que vem do
log. Trocar o custo para MSE quebra o cancelamento — e é por isso que o gradiente
do MSE ganha um $\sigma'(z)$ sobrando, que morre nas pontas. O gráfico que você
viu na seção anterior é esse termo.

### Exercício 4 — `gradiente(X, y, w, b)`

Devolva `(dw, db)`, onde `dw` tem o mesmo formato de `w`.

Dica: `erro = p - y` é um vetor de tamanho `m`. Para o `dw`, você precisa
multiplicar cada erro pela linha correspondente de `X` e somar — que é
`X.T @ erro / m`, o mesmo do desafio da revisão da A2.


In [ ]:
def gradiente(X, y, w, b):
    """Derivadas do custo em relação a w e a b.

    retorna: (dw, db) — dw com o formato de w, db um número
    """
    ### SEU CÓDIGO AQUI ###  (≈ 4 linhas)
    raise NotImplementedError("Apague esta linha e devolva (dw, db).")


In [ ]:
# O teste de ouro: comparar com a derivada numérica.
# Se você mexe w[j] por um épsilon minúsculo, o custo muda por
# (derivada × épsilon). Isso não depende de nenhuma álgebra ter sido feita
# certo — é a definição de derivada. É assim que se caça erro de gradiente
# em código de verdade.
def gradiente_numerico(X, y, w, b, eps=1e-6):
    dw = np.zeros_like(w, dtype=float)
    for j in range(len(w)):
        mais, menos = w.astype(float).copy(), w.astype(float).copy()
        mais[j] += eps; menos[j] -= eps
        dw[j] = (custo(X, y, mais, b) - custo(X, y, menos, b)) / (2 * eps)
    db = (custo(X, y, w, b + eps) - custo(X, y, w, b - eps)) / (2 * eps)
    return dw, db

w_g, b_g = np.array([0.5, -0.25]), 0.1
dw_meu, db_meu = gradiente(X_c, y_c, w_g, b_g)
dw_num, db_num = gradiente_numerico(X_c, y_c, w_g, b_g)

dw0, db0 = gradiente(X_c, y_c, np.zeros(2), 0.0)

verificar(
    "Exercício 4 · gradiente",
    ("dw tem o formato de w", np.shape(dw_meu), (2,)),
    ("db é um número", np.ndim(db_meu), 0),
    ("dw bate com a derivada numérica", dw_meu, dw_num),
    ("db bate com a derivada numérica", db_meu, db_num),
    ("com w=0, db é a média de (0,5 − y)", db0, 0.5 - y_c.mean()),
    ("no ótimo o gradiente é ~0",
     bool(np.abs(gradiente(np.array([[1.0], [-1.0]]), np.array([1.0, 0.0]),
                           np.array([50.0]), 0.0)[1]) < 1e-15)),
)


<details><summary>Resposta do Exercício 4</summary>

```python
    p = prever_probabilidade(X, w, b)
    erro = p - y
    dw = X.T @ erro / len(y)
    db = erro.mean()
    return dw, db
```

**O `X.T @ erro` é o passo que confunde.** `X` é (m, n) e `erro` é (m,).
Transpondo, `X.T` é (n, m), e o produto com `erro` dá (n,) — um número por
feature, que é exatamente o formato de `dw`. Por dentro, a coordenada `j` desse
resultado é $\sum_i x_j^{(i)}\,\text{erro}^{(i)}$: cada erro pesado pela feature
`j` daquele exemplo, somado. É a fórmula do enunciado, escrita sem laço.

**Sobre o teste numérico:** ele é a ferramenta que você usa quando o treino não
converge e você não sabe se o bug está no gradiente ou no resto. Se o analítico
e o numérico discordam, o erro é seu; se concordam e mesmo assim não treina, o
problema é outro (taxa de aprendizado, escala, dado). Vale guardar a receita.

</details>


## 6. O treino

Com custo e gradiente, o loop é o mesmo da Aula 1 — literalmente o mesmo. É por
isso que vale ter escrito aquele: você já sabe este.

### Exercício 5 — `treinar(X, y, alpha, n_passos)`

Comece com `w` zerado e `b = 0`. A cada passo: calcule o gradiente, ande contra
ele, e **guarde o custo** numa lista para dar para plotar depois.


In [ ]:
def treinar(X, y, alpha=0.1, n_passos=2000):
    """Gradient descent na regressão logística.

    retorna: (w, b, historico_de_custo)
    """
    w = np.zeros(X.shape[1])
    b = 0.0
    historico = []
    ### SEU CÓDIGO AQUI ###  (≈ 5 linhas)
    raise NotImplementedError("Apague esta linha e implemente o loop.")


In [ ]:
w_t, b_t, hist = treinar(X_treino, y_treino, alpha=0.5, n_passos=3000)

# O gabarito vem do sklearn, que resolve o mesmo problema por outro caminho.
from sklearn.linear_model import LogisticRegression
ref = LogisticRegression(C=1e6, max_iter=10000).fit(X_treino, y_treino)

acuracia = ((prever_probabilidade(X_teste, w_t, b_t) >= 0.5).astype(int) == y_teste).mean()

verificar(
    "Exercício 5 · treinar",
    ("devolve três coisas", len(treinar(X_treino, y_treino, 0.5, 5)), 3),
    ("guardou um custo por passo", len(hist), 3000),
    ("o custo caiu", bool(hist[-1] < hist[0])),
    ("o custo cai de forma monótona (alpha razoável)",
     bool(np.all(np.diff(hist) <= 1e-9))),
    ("chegou a 2% do gabarito do sklearn (pesos)",
     bool(np.allclose(w_t, ref.coef_[0], rtol=0.02))),
    ("chegou a 2% do gabarito do sklearn (viés)",
     bool(np.isclose(b_t, ref.intercept_[0], rtol=0.02))),
    ("acurácia no teste acima de 85%", bool(acuracia > 0.85)),
)
print(f"\nacurácia no teste: {acuracia:.1%}")


<details><summary>Resposta do Exercício 5</summary>

```python
    for _ in range(n_passos):
        dw, db = gradiente(X, y, w, b)
        w = w - alpha * dw
        b = b - alpha * db
        historico.append(custo(X, y, w, b))
    return w, b, historico
```

**Uma escolha escondida aí:** o custo é guardado *depois* do passo, não antes.
Faz diferença só no primeiro ponto do gráfico, mas se você registrar antes, o
histórico tem o custo inicial e nunca o final — e aí "o último custo" que você
imprime não é o do modelo que você devolveu.

**Se o teste da monotonicidade falhou** e os outros passaram, o `alpha` está
grande demais: o custo desce em ziguezague em vez de descer liso. É o mesmo
diagnóstico da Aula 2, agora numa superfície diferente.

</details>


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(hist, c="tab:green"); ax[0].set_xlabel("passo"); ax[0].set_ylabel("custo")
ax[0].set_title("O custo ao longo do treino")

# A fronteira aprendida: onde w·x + b = 0, ou seja, onde p = 0,5
x1g = np.linspace(X_treino[:, 0].min() - 0.5, X_treino[:, 0].max() + 0.5, 100)
x2g = -(w_t[0] * x1g + b_t) / w_t[1]
for classe, cor, nome in [(0, "tab:blue", "benigno"), (1, "tab:red", "maligno")]:
    m = y_treino == classe
    ax[1].scatter(X_treino[m, 0], X_treino[m, 1], s=12, c=cor, label=nome, alpha=0.6)
ax[1].plot(x1g, x2g, c="k", lw=2, label="fronteira (p = 0,5)")
ax[1].set_ylim(X_treino[:, 1].min() - 0.5, X_treino[:, 1].max() + 0.5)
ax[1].set_xlabel(COLUNAS[0]); ax[1].set_ylabel(COLUNAS[1])
ax[1].set_title("A fronteira que o seu código achou"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"seus pesos:      w = {w_t},  b = {b_t:.4f}")
print(f"gabarito sklearn: w = {ref.coef_[0]},  b = {ref.intercept_[0]:.4f}")


## 7. Quando a reta não serve

A página mostra o caso da bolha dentro do anel: nenhuma reta separa um grupo que
está *dentro* do outro. A saída, como na Aula 2, é **inventar features**.

Rode a célula abaixo e veja o modelo que você escreveu fracassar honestamente.


In [ ]:
from sklearn.datasets import make_circles

X_circ, y_circ = make_circles(n_samples=400, factor=0.45, noise=0.08, random_state=1)

w_c, b_c, _ = treinar(X_circ, y_circ, alpha=0.5, n_passos=2000)
acc_reta = ((prever_probabilidade(X_circ, w_c, b_c) >= 0.5).astype(int) == y_circ).mean()
print(f"acurácia com as 2 features cruas: {acc_reta:.1%}  (chutar daria 50%)")


### Exercício 6 — `features_polinomiais(X)`

Receba `X` de formato (m, 2) com as colunas $x_1$ e $x_2$, e devolva (m, 5) com

$$[\,x_1,\; x_2,\; x_1^2,\; x_2^2,\; x_1x_2\,]$$

nessa ordem. Dica: `np.column_stack`.

Pense por que isso resolve: a fronteira continua sendo
$\mathbf{w}\cdot\mathbf{x} + b = 0$, mas agora com $x_1^2$ e $x_2^2$ entre as
features, essa equação **é** a de um círculo. A fronteira nunca deixou de ser
reta — ela só é reta num espaço de 5 dimensões, e o que vemos é a sombra dela.


In [ ]:
def features_polinomiais(X):
    """(m, 2) → (m, 5): x1, x2, x1², x2², x1·x2."""
    x1, x2 = X[:, 0], X[:, 1]
    ### SEU CÓDIGO AQUI ###  (≈ 1 linha)
    raise NotImplementedError("Apague esta linha e devolva as 5 colunas.")


In [ ]:
X_f = np.array([[2.0, 3.0], [-1.0, 4.0]])
saida = features_polinomiais(X_f)

verificar(
    "Exercício 6 · features_polinomiais",
    ("o formato é (m, 5)", np.shape(saida), (2, 5)),
    ("primeira linha: [2, 3, 4, 9, 6]", saida[0], [2.0, 3.0, 4.0, 9.0, 6.0]),
    ("segunda linha: [-1, 4, 1, 16, -4]", saida[1], [-1.0, 4.0, 1.0, 16.0, -4.0]),
    ("as duas primeiras colunas são o X original", saida[:, :2], X_f),
    ("roda no dataset dos círculos", np.shape(features_polinomiais(X_circ)), (400, 5)),
)


<details><summary>Resposta do Exercício 6</summary>

```python
    return np.column_stack([x1, x2, x1**2, x2**2, x1 * x2])
```

**Por que `x1 * x2` precisa estar lá.** Com só os quadrados, a fronteira é um
círculo ou uma elipse *alinhada aos eixos*. O termo cruzado é o que permite
girar a elipse. Neste dataset os círculos já estão centrados, então ele quase
não faz falta — mas tire-o e teste num dado inclinado que a diferença aparece.

**E por que ninguém escreve isso na mão em produção:** o
`PolynomialFeatures(degree=2)` do sklearn, da Aula 2, gera exatamente estas
colunas. Escrever à mão uma vez é para ver que não há mágica nenhuma ali dentro.

</details>


In [ ]:
w_pc, b_pc, _ = treinar(features_polinomiais(X_circ), y_circ, alpha=0.5, n_passos=4000)
acc_poli = ((prever_probabilidade(features_polinomiais(X_circ), w_pc, b_pc) >= 0.5)
            .astype(int) == y_circ).mean()

print(f"com as 2 features cruas : {acc_reta:.1%}")
print(f"com as 5 features       : {acc_poli:.1%}")

g = np.linspace(-1.6, 1.6, 300)
G1, G2 = np.meshgrid(g, g)
grade = np.column_stack([G1.ravel(), G2.ravel()])
P = prever_probabilidade(features_polinomiais(grade), w_pc, b_pc).reshape(G1.shape)

fig, ax = plt.subplots(figsize=(4.8, 4.4))
ax.contourf(G1, G2, P, levels=[0, 0.5, 1], colors=["#dbeafe", "#fee2e2"], alpha=0.9)
ax.contour(G1, G2, P, levels=[0.5], colors="k", linewidths=2)
for classe, cor in [(0, "tab:blue"), (1, "tab:red")]:
    m = y_circ == classe
    ax.scatter(X_circ[m, 0], X_circ[m, 1], s=10, c=cor, alpha=0.7)
ax.set_title("A fronteira virou um círculo")
plt.tight_layout(); plt.show()


## 8. As métricas — onde a acurácia mente

Última parte da página, e a mais importante na prática. Primeiro, a armadilha:


In [ ]:
# Um problema desbalanceado, como a fraude do exemplo da página: 1% de positivos
rng2 = np.random.default_rng(7)
n = 2000
y_raro = (rng2.random(n) < 0.01).astype(int)
previsao_burra = np.zeros(n, dtype=int)   # "não é fraude" para tudo

print(f"positivos reais: {y_raro.sum()} em {n}  ({y_raro.mean():.1%})")
print(f"acurácia do modelo que nunca acusa nada: {(previsao_burra == y_raro).mean():.1%}")
print("\nE ele não pega uma fraude sequer. É por isso que acurácia sozinha não serve.")


### Exercício 7 — `matriz_confusao(y, y_previsto)`

Devolva `(vp, fp, fn, vn)`, com a classe 1 como positiva:

- **vp** — era 1 e o modelo disse 1
- **fp** — era 0 e o modelo disse 1 (alarme falso)
- **fn** — era 1 e o modelo disse 0 (deixou passar)
- **vn** — era 0 e o modelo disse 0

Dica: `((y == 1) & (y_previsto == 1)).sum()` — sem `for`.


In [ ]:
def matriz_confusao(y, y_previsto):
    """Devolve (vp, fp, fn, vn) como inteiros."""
    y = np.asarray(y).astype(int)
    y_previsto = np.asarray(y_previsto).astype(int)
    ### SEU CÓDIGO AQUI ###  (≈ 4 linhas)
    raise NotImplementedError("Apague esta linha e devolva (vp, fp, fn, vn).")


In [ ]:
y_v = np.array([1, 1, 1, 0, 0, 0, 0, 1])
p_v = np.array([1, 1, 0, 1, 0, 0, 0, 0])
#               ✓  ✓  fn fp ✓  ✓  ✓  fn   → vp=2, fp=1, fn=2, vn=3

verificar(
    "Exercício 7 · matriz_confusao",
    ("devolve quatro números", len(matriz_confusao(y_v, p_v)), 4),
    ("o caso do enunciado dá (2, 1, 2, 3)", matriz_confusao(y_v, p_v), (2, 1, 2, 3)),
    ("as quatro caixas somam o total", sum(matriz_confusao(y_v, p_v)), len(y_v)),
    ("acertando tudo, fp e fn são zero", matriz_confusao(y_v, y_v)[1:3], (0, 0)),
    ("o modelo que nunca acusa nada tem vp = 0",
     matriz_confusao(y_raro, previsao_burra)[0], 0),
)


<details><summary>Resposta do Exercício 7</summary>

```python
    vp = int(((y == 1) & (y_previsto == 1)).sum())
    fp = int(((y == 0) & (y_previsto == 1)).sum())
    fn = int(((y == 1) & (y_previsto == 0)).sum())
    vn = int(((y == 0) & (y_previsto == 0)).sum())
    return vp, fp, fn, vn
```

**A ordem dos nomes é a armadilha.** "Falso positivo" descreve o que o **modelo
disse** (positivo) e diz que ele estava errado (falso). Então FP é
`era 0, disse 1`. Se você trocou FP com FN, é quase certo que leu como "falso" =
"o valor real" — todo mundo faz isso uma vez.

</details>


### Exercício 8 — `precision_recall(y, y_previsto)`

Da matriz saem os dois números que importam:

$$\text{precision} = \frac{VP}{VP + FP} \qquad \text{recall} = \frac{VP}{VP + FN}$$

Precision parte **do que o modelo disse** ("dos meus alarmes, quantos
prestavam?"). Recall parte **do que existia** ("dos casos reais, quantos eu
peguei?").

**Cuidado com o denominador zero:** um modelo que nunca acusa nada tem
$VP + FP = 0$, e a precision fica indefinida. Devolva `0.0` nesse caso, em vez
de deixar estourar.


In [ ]:
def precision_recall(y, y_previsto):
    """Devolve (precision, recall) como floats. Denominador zero → 0.0."""
    vp, fp, fn, vn = matriz_confusao(y, y_previsto)
    ### SEU CÓDIGO AQUI ###  (≈ 3 linhas)
    raise NotImplementedError("Apague esta linha e devolva (precision, recall).")


In [ ]:
verificar(
    "Exercício 8 · precision_recall",
    ("o caso do exercício 7: precision = 2/3", precision_recall(y_v, p_v)[0], 2 / 3),
    ("o caso do exercício 7: recall = 2/4", precision_recall(y_v, p_v)[1], 0.5),
    ("acertando tudo, os dois são 1", precision_recall(y_v, y_v), (1.0, 1.0)),
    ("o modelo que nunca acusa: recall = 0",
     precision_recall(y_raro, previsao_burra)[1], 0.0),
    ("o modelo que nunca acusa: precision = 0, e não estoura",
     precision_recall(y_raro, previsao_burra)[0], 0.0),
    ("acusando tudo, o recall é 1",
     precision_recall(y_v, np.ones_like(y_v))[1], 1.0),
)


<details><summary>Resposta do Exercício 8</summary>

```python
    precision = vp / (vp + fp) if (vp + fp) > 0 else 0.0
    recall    = vp / (vp + fn) if (vp + fn) > 0 else 0.0
    return precision, recall
```

**Por que devolver 0 e não `nan`.** É uma escolha, não uma verdade: um modelo
que nunca acusa nada não tem precision *definida* — ele não fez nenhum alarme
para se julgar. Devolver 0 é a convenção do sklearn (que ainda emite um aviso), e
serve porque a alternativa, `nan`, contamina qualquer média que você calcule
depois. Só não confunda "precision 0" com "todos os alarmes erraram": aqui é
"não houve alarme".

</details>


### O limiar, e os dois brigando

Agora o ponto da página: **precision e recall andam em direções opostas** quando
você mexe no limiar. Com o seu classificador treinado, dá para ver isso no dado
de verdade.


In [ ]:
p_teste = prever_probabilidade(X_teste, w_t, b_t)
limiares = np.linspace(0.02, 0.98, 97)
precs, recs = [], []
for t in limiares:
    pr, rc = precision_recall(y_teste, (p_teste >= t).astype(int))
    precs.append(pr); recs.append(rc)

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(limiares, precs, label="precision", c="tab:blue")
ax.plot(limiares, recs, label="recall", c="tab:orange")
ax.axvline(0.5, ls=":", c="gray")
ax.set_xlabel("limiar"); ax.set_ylabel("valor"); ax.legend()
ax.set_title("Um sobe quando o outro desce")
plt.tight_layout(); plt.show()

for t in [0.2, 0.5, 0.8]:
    pr, rc = precision_recall(y_teste, (p_teste >= t).astype(int))
    vp, fp, fn, vn = matriz_confusao(y_teste, (p_teste >= t).astype(int))
    print(f"limiar {t:.1f}:  precision {pr:.2f}   recall {rc:.2f}   "
          f"(deixou passar {fn} tumores malignos, deu {fp} alarmes falsos)")


🔬 **Experimente.** Volte na célula acima e olhe a coluna do `fn` — cada unidade
ali é um tumor maligno que o modelo mandou para casa. Agora releia a frase da
página: *"diagnóstico de câncer quer recall alto — deixar um doente passar é
grave, um alarme falso é só um exame a mais"*. O limiar 0,5 não tem nada de
especial, e nesse problema ele é provavelmente alto demais.

---

## 🔬 Desafio

**1.** Escreva `limiar_para_recall(y, p, recall_minimo)`: dado o vetor de
probabilidades, ache o **maior** limiar que ainda garante o recall pedido. Maior
porque, entre todos os que atendem a exigência, ele é o que dá menos alarme
falso. Use com `recall_minimo=0.98` e veja quantos falsos positivos custa.

<details><summary>Resposta</summary>

```python
def limiar_para_recall(y, p, recall_minimo):
    candidatos = np.unique(np.concatenate([p, [0.0]]))
    validos = [t for t in candidatos
               if precision_recall(y, (p >= t).astype(int))[1] >= recall_minimo]
    return max(validos) if validos else 0.0

t = limiar_para_recall(y_teste, p_teste, 0.98)
pr, rc = precision_recall(y_teste, (p_teste >= t).astype(int))
vp, fp, fn, vn = matriz_confusao(y_teste, (p_teste >= t).astype(int))
print(f"limiar {t:.3f} → recall {rc:.2f}, precision {pr:.2f}, {fp} alarmes falsos")
```

Os candidatos saem dos **próprios valores previstos**, não de uma grade fixa: o
recall só muda quando o limiar cruza a probabilidade de algum exemplo, então
qualquer valor entre dois vizinhos dá o mesmo resultado. Varrer `linspace` também
funciona, mas pode passar batido pelo ponto exato.

</details>

**2.** A acurácia trata errar um maligno e errar um benigno como a mesma coisa.
Escreva um "custo clínico" que pese um falso negativo **10 vezes** mais que um
falso positivo, e ache o limiar que o minimiza. Ele fica acima ou abaixo de 0,5?

<details><summary>Resposta</summary>

```python
def custo_clinico(y, p, t, peso_fn=10):
    vp, fp, fn, vn = matriz_confusao(y, (p >= t).astype(int))
    return peso_fn * fn + fp

melhor = min(limiares, key=lambda t: custo_clinico(y_teste, p_teste, t))
print(f"limiar que minimiza o custo clínico: {melhor:.2f}")
```

Fica **abaixo** de 0,5, e é o ponto inteiro do exercício: o limiar ótimo não sai
do modelo, sai de quanto custa cada tipo de erro **no seu problema**. Dois
hospitais com o mesmo modelo e políticas diferentes escolhem limiares diferentes,
e os dois estão certos.

</details>

**3.** Treine de novo usando as **30 features** do dataset (`dados.data` inteiro,
padronizado), em vez de duas. O que acontece com o recall no teste? E o custo do
treino cai mais rápido ou mais devagar?

<details><summary>Resposta</summary>

```python
Xt30 = dados.data[tr]; Xe30 = dados.data[te]
m30, d30 = Xt30.mean(axis=0), Xt30.std(axis=0)
Xt30, Xe30 = (Xt30 - m30) / d30, (Xe30 - m30) / d30

w30, b30, h30 = treinar(Xt30, y_treino, alpha=0.5, n_passos=3000)
p30 = prever_probabilidade(Xe30, w30, b30)
print("2 features :", precision_recall(y_teste, (p_teste >= 0.5).astype(int)))
print("30 features:", precision_recall(y_teste, (p30 >= 0.5).astype(int)))
```

Melhora bastante — mas note que isso **não contradiz** o quiz da Aula 2 sobre
features inúteis. Lá as 40 colunas extras eram ruído; aqui as 28 a mais são
medidas reais do mesmo tumor, feitas por quem sabia o que estava medindo. Feature
entra por merecimento, e estas merecem.

</details>

---

## O que ficou

- **A sigmoid é a inversa do log-odds**, não um chute com formato de S. Assumir
  que o log das chances é linear te dá ela de graça.
- **Cross-entropy, não MSE** — e o motivo é a forma da superfície: uma é tigela,
  a outra tem platôs onde o gradiente morre.
- **O gradiente é idêntico ao da regressão linear.** Custo diferente, modelo
  diferente, mesma fórmula — porque a sigmoid e a cross-entropy foram feitas uma
  para a outra.
- **Fronteira curva continua sendo modelo linear**, igual à Aula 2: muda quais
  features entram, não o algoritmo.
- **Acurácia mente em dado desbalanceado.** Precision e recall brigam, e quem
  decide o limiar é o custo do erro no seu problema — não o modelo.

**Anterior:** [Aula 2 — Escalando o Modelo](https://colab.research.google.com/github/AlexChequer/notebooks-insperai/blob/main/trainees/aula-02-escalando-o-modelo.ipynb)
